<a href="https://colab.research.google.com/github/SpikemenDan/vawt-research/blob/main/Aarion's_Modified_Code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
#@title Block 1: Master Initialization & Historic Data {display-mode: "form"}
# ==============================================================================
import os
import glob
import pandas as pd
import numpy as np
from google.colab import drive

# Smart Drive Mount
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
else:
    print("✅ Google Drive already mounted.")

MASTER_DIR = '/content/drive/MyDrive/TurbineResearch'
if not os.path.exists(MASTER_DIR):
    os.makedirs(MASTER_DIR)

def save_and_track_matrix(df, file_name, directory=MASTER_DIR):
    if not file_name.endswith('.parquet'):
        file_name += '.parquet'
    full_path = os.path.join(directory, file_name)
    df.to_parquet(full_path, engine='auto', index=False)
    var_name = file_name.replace('.parquet', '').replace(' ', '_').lower()
    print(f"SUCCESS: {var_name} saved to Drive.")

# Track BOTH peaks and raw sweeps to prevent data loss
historic_df_list = []
historic_raw_list = []

# ==============================================================================
# 1. LOCALIZED PEAK EXTRACTION: HUNT 2024 DATASET
# ==============================================================================
hunt_parquet_path = os.path.join(MASTER_DIR, 'hunt_kinematic_master.parquet')

if os.path.exists(hunt_parquet_path):
    print("Loading processed Hunt .parquet file...")
    df_hunt_raw = pd.read_parquet(hunt_parquet_path)

    if 'dataset_source' not in df_hunt_raw.columns:
        df_hunt_raw['dataset_source'] = 'Hunt 2024'

    re_targets = [0.75e5, 1.6e5, 2.7e5, 8.3e5]
    def snap_to_nearest(val, targets):
        return min(targets, key=lambda x: abs(x - val))

    df_hunt_raw['Re_D_binned'] = df_hunt_raw['Re_D'].apply(lambda x: snap_to_nearest(x, re_targets))

    # STRICT FIX: Explicitly include alpha_p to prevent sawtooth grouping
    df_hunt_raw['label'] = "Hunt: N=" + df_hunt_raw['N'].astype(int).astype(str) + \
                           ", c/R=" + df_hunt_raw['cToR'].round(2).astype(str) + \
                           ", Re=" + df_hunt_raw['Re_D_binned'].apply(lambda x: f"{x:.1e}") + \
                           ", alpha_p=" + df_hunt_raw['alpha_p'].astype(str)

    hunt_group_cols = ['N', 'cToR', 'alpha_p', 'Re_D_binned']
    peak_indices = df_hunt_raw.groupby(hunt_group_cols)['Cp'].idxmax()
    df_hunt_peaks = df_hunt_raw.loc[peak_indices].copy()

    if 'Dynamic_Flow_Opacity' not in df_hunt_peaks.columns and 'solidity' in df_hunt_peaks.columns:
        df_hunt_peaks['Dynamic_Flow_Opacity'] = df_hunt_peaks['solidity'] * df_hunt_peaks['TSR']
        df_hunt_raw['Dynamic_Flow_Opacity'] = df_hunt_raw['solidity'] * df_hunt_raw['TSR']

    historic_df_list.append(df_hunt_peaks)
    historic_raw_list.append(df_hunt_raw)

    print(f"✅ Ingested Hunt dataset.")
    print(f"   - Raw sweep points added: {len(df_hunt_raw)}")
    print(f"   - Localized absolute peaks: {len(df_hunt_peaks)} based on {hunt_group_cols}.")
else:
    print(f"⚠️ Hunt dataset not found at {hunt_parquet_path}.")

# ==============================================================================
# 2. LOCALIZED PEAK EXTRACTION: UNH-RVAT DATASET
# ==============================================================================
unh_parquet_path = os.path.join(MASTER_DIR, 'unh_kinematic_master.parquet')

if os.path.exists(unh_parquet_path):
    print("\nLoading processed UNH-RVAT .parquet file...")
    df_unh_raw = pd.read_parquet(unh_parquet_path)

    if 'dataset_source' not in df_unh_raw.columns:
        df_unh_raw['dataset_source'] = 'UNH-RVAT'

    df_unh_raw = df_unh_raw.rename(columns={'mean_tsr': 'TSR', 'mean_cp': 'Cp'})

    if 'cToR' not in df_unh_raw.columns:
        df_unh_raw['cToR'] = 0.14 / 0.5
    if 'N' not in df_unh_raw.columns:
        df_unh_raw['N'] = 3
    if 'alpha_p' not in df_unh_raw.columns:
        df_unh_raw['alpha_p'] = 0.0

    if 'solidity' not in df_unh_raw.columns:
        df_unh_raw['solidity'] = (df_unh_raw['N'] * df_unh_raw['cToR']) / (2 * np.pi)
    if 'Dynamic_Flow_Opacity' not in df_unh_raw.columns:
        df_unh_raw['Dynamic_Flow_Opacity'] = df_unh_raw['solidity'] * df_unh_raw['TSR']

    if 'mean_tow_speed' in df_unh_raw.columns:
        df_unh_raw['tow_speed_binned'] = df_unh_raw['mean_tow_speed'].round(1)
        unh_group_cols = ['tow_speed_binned']
    else:
        unh_group_cols = ['source_file'] if 'source_file' in df_unh_raw.columns else ['dataset_source']

    # Explicit Root Label Generation for UNH separating the 5 sweeps
    df_unh_raw['label'] = "UNH-RVAT: U=" + df_unh_raw['tow_speed_binned'].astype(str) + " m/s"

    peak_indices_unh = df_unh_raw.groupby(unh_group_cols)['Cp'].idxmax()
    df_unh_peaks = df_unh_raw.loc[peak_indices_unh].copy()

    historic_df_list.append(df_unh_peaks)
    historic_raw_list.append(df_unh_raw)

    print(f"✅ Ingested UNH-RVAT dataset.")
    print(f"   - Raw sweep points added: {len(df_unh_raw)}")
    print(f"   - Localized absolute peaks: {len(df_unh_peaks)} based on {unh_group_cols}.")
else:
    print(f"⚠️ UNH dataset not found at {unh_parquet_path}.")

✅ Google Drive already mounted.
Loading processed Hunt .parquet file...
✅ Ingested Hunt dataset.
   - Raw sweep points added: 4083
   - Localized absolute peaks: 223 based on ['N', 'cToR', 'alpha_p', 'Re_D_binned'].

Loading processed UNH-RVAT .parquet file...
✅ Ingested UNH-RVAT dataset.
   - Raw sweep points added: 125
   - Localized absolute peaks: 5 based on ['tow_speed_binned'].


In [18]:
#@title Block 2: Paper Grouping + Legend Panel {display-mode: "form"}
# ==============================================================================
import re
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.colors as mcolors
import ipywidgets as widgets

# ------------------------------------------------------------------------------
# MANUAL MERGES (optional)
# If two names still show up as separate papers, map them here.
# Matching ignores capitalization and spacing.
#   PAPER_ALIASES = {"Howell 32": "Howell et al. [32]", "howell et al 32": "Howell et al. [32]"}
# ------------------------------------------------------------------------------
PAPER_ALIASES = {}

_TRAILING_SWEEP_TAG = re.compile(
    r"[\s,;:_\-–]*\b(?:fig(?:ure)?\.?\s*\d+\w*|N\s*=.*|cToR\s*=.*|C-R\s*=.*|Re(?:_D)?\s*=.*|alpha_p\s*=.*)$",
    re.IGNORECASE,
)


def _norm_key(text):
    return re.sub(r"\s+", " ", str(text).replace("\xa0", " ")).strip().lower()


def _clean_paper_name(name):
    """'Howell et al.\xa0[32] (N=3)' -> 'Howell et al. [32]'"""
    s = str(name).replace("\xa0", " ")
    s = re.sub(r"\s+", " ", s).strip()
    s = s.split("(")[0].strip()
    s = _TRAILING_SWEEP_TAG.sub("", s).strip(" ,;:_-–")
    return s or "Unknown source"


def prepare_frames(frames, verbose=True):
    """
    Shared cleanup used by BOTH action cells so they always agree.
      - merges messy dataset names into one name per paper
      - guarantees N / cToR / alpha_p / Re_D / opacity columns exist
      - gives every curve a unique, readable `label` (and `curve_name` for the legend panel)
    Pass a list of dataframes (e.g. [df_raw, df_peaks]); returns the cleaned list.
    """
    frames = [f.copy() for f in frames]
    aliases = {_norm_key(k): v for k, v in PAPER_ALIASES.items()}

    # ---- 1. one canonical name per paper ----
    raw_names = pd.unique(pd.concat([f["dataset_source"] for f in frames]).dropna().astype(str))
    resolved = {}
    for raw in raw_names:
        cleaned = _clean_paper_name(raw)
        resolved[raw] = aliases.get(_norm_key(raw)) or aliases.get(cleaned.lower()) or cleaned

    canon = {}
    for name in sorted(set(resolved.values())):
        canon.setdefault(name.lower(), name)
    final = {raw: canon[name.lower()] for raw, name in resolved.items()}

    for f in frames:
        f["original_dataset_source"] = f["dataset_source"]
        f["dataset_source"] = f["dataset_source"].astype(object).map(
            lambda x: final.get(str(x), "Unknown source") if pd.notna(x) else "Unknown source"
        )

        if "Dynamic_Flow_Opacity" not in f.columns and "Dynamic_Blockage" in f.columns:
            f["Dynamic_Flow_Opacity"] = f["Dynamic_Blockage"]
        elif "Dynamic_Flow_Opacity" in f.columns and "Dynamic_Blockage" not in f.columns:
            f["Dynamic_Blockage"] = f["Dynamic_Flow_Opacity"]

        for col in ["N", "cToR", "alpha_p", "Re_D"]:
            if col not in f.columns:
                f[col] = np.nan
            f[col] = pd.to_numeric(f[col], errors="coerce")

        # identity of a curve: config_id for uploaded data, existing label for Hunt / UNH
        if "label" not in f.columns:
            f["label"] = np.nan
        f["label"] = f["label"].astype(object)
        if "config_id" in f.columns:
            has_id = f["config_id"].notna()
            f.loc[has_id, "label"] = f.loc[has_id, "config_id"].astype(str)
        missing = f["label"].isna()
        if missing.any():
            f.loc[missing, "label"] = "params:" + f.loc[missing, ["N", "cToR", "alpha_p", "Re_D"]].astype(str).agg("|".join, axis=1)
        f["_orig"] = f["label"].astype(str)

    # ---- 2. readable + unique label per curve ----
    uniq = pd.concat(
        [f[["dataset_source", "_orig", "N", "cToR", "alpha_p", "Re_D"]] for f in frames]
    ).drop_duplicates(["dataset_source", "_orig"])

    def describe(r):
        orig = str(r["_orig"])
        body = orig.split(": ", 1)[1] if ": " in orig else orig
        if ": " in orig and "=" in body:          # already readable (Hunt / UNH style)
            return body
        parts = []
        if pd.notna(r["N"]):       parts.append(f"N={int(r['N'])}")
        if pd.notna(r["cToR"]):    parts.append(f"c/R={r['cToR']:.4g}")
        if pd.notna(r["Re_D"]):    parts.append(f"Re={r['Re_D']:.1e}")
        if pd.notna(r["alpha_p"]): parts.append(f"alpha_p={r['alpha_p']:g}")
        return ", ".join(parts) if parts else body

    uniq = uniq.sort_values(["dataset_source", "_orig"]).reset_index(drop=True)
    uniq["_name"] = uniq.apply(describe, axis=1)
    grp = uniq.groupby(["dataset_source", "_name"])["_orig"]
    dup_size = grp.transform("size")
    dup_rank = grp.cumcount() + 1
    uniq["_name"] = np.where(dup_size > 1, uniq["_name"] + " #" + dup_rank.astype(str), uniq["_name"])
    uniq["_label"] = uniq["dataset_source"] + ": " + uniq["_name"]

    mapping = {(p, o): (lab, nm) for p, o, lab, nm in zip(uniq["dataset_source"], uniq["_orig"], uniq["_label"], uniq["_name"])}
    for f in frames:
        pairs = [mapping[k] for k in zip(f["dataset_source"], f["_orig"])]
        f["label"] = [p[0] for p in pairs]
        f["curve_name"] = [p[1] for p in pairs]
        f.drop(columns=["_orig"], inplace=True)

    if verbose:
        groups = {}
        for raw, paper in final.items():
            groups.setdefault(paper, []).append(raw)
        print(f"📚 {len(groups)} papers detected (from {len(final)} raw dataset names).")
        for paper in sorted(groups):
            raws = sorted(set(groups[paper]))
            if len(raws) > 1:
                shown = "; ".join(repr(r) for r in raws[:4]) + (" …" if len(raws) > 4 else "")
                print(f"   • {paper}  ←  merged {len(raws)} names: {shown}")

    return frames


_MARKER_OPTIONS = ["o", "X", "s", "^", "D", "v", "P", "*"]
_MARKER_GLYPH = {"o": "●", "X": "✖", "s": "■", "^": "▲", "D": "◆", "v": "▼", "P": "✚", "*": "★"}


def make_style_dicts(papers):
    papers = list(papers)
    pal = sns.color_palette("tab10" if len(papers) <= 10 else "tab20", max(len(papers), 1))
    palette_dict = {p: pal[i % len(pal)] for i, p in enumerate(papers)}
    marker_dict = {p: _MARKER_OPTIONS[i % len(_MARKER_OPTIONS)] for i, p in enumerate(papers)}
    return palette_dict, marker_dict


class LegendSelector:
    """
    Side panel that doubles as a legend + filter.
      - ONE checkbox per paper: turns every curve of that paper on/off
      - click ▸ to expand a paper and isolate individual curves
      - 'solo' = show only that paper; 'Show all' / 'Hide all' on top
    """

    def __init__(self, frames, palette_dict, marker_dict):
        curves = pd.concat(
            [f[["dataset_source", "label", "curve_name", "N", "cToR", "alpha_p", "Re_D"]] for f in frames]
        ).drop_duplicates(["dataset_source", "label"])

        self.papers = sorted(curves["dataset_source"].unique())
        self._callbacks = []
        self._busy = False
        self.master = {}
        self.curve_cbs = {}
        self._counts = {}
        self._bodies = {}

        rows = []
        for paper in self.papers:
            sub = curves[curves["dataset_source"] == paper].sort_values(
                ["N", "cToR", "alpha_p", "Re_D", "label"], na_position="last"
            )
            color = mcolors.to_hex(palette_dict.get(paper, (0.5, 0.5, 0.5)))
            glyph = _MARKER_GLYPH.get(marker_dict.get(paper, "o"), "●")
            swatch = widgets.HTML(
                f"<span style='color:{color};font-size:17px;line-height:26px'>{glyph}</span>",
                layout=widgets.Layout(width="22px"),
            )

            cbs = {}
            for lab, name in zip(sub["label"], sub["curve_name"]):
                cbs[lab] = widgets.Checkbox(
                    value=True, description=name, indent=False,
                    layout=widgets.Layout(width="max-content"),
                )
            self.curve_cbs[paper] = cbs
            self._counts[paper] = len(cbs)

            master = widgets.Checkbox(
                value=True, indent=False,
                style={"description_width": "initial"},
                layout=widgets.Layout(width="auto", flex="1 1 auto"),
            )
            self.master[paper] = master

            expand = widgets.ToggleButton(
                value=False, description="▸", tooltip="Show / hide individual curves",
                layout=widgets.Layout(width="34px"),
            )
            solo = widgets.Button(
                description="solo", tooltip="Show only this paper",
                layout=widgets.Layout(width="52px"),
            )
            body = widgets.VBox(
                list(cbs.values()),
                layout=widgets.Layout(display="none", margin="0 0 6px 34px", max_height="240px", overflow="auto"),
            )
            self._bodies[paper] = body

            self._wire(paper, master, expand, solo, body)
            self._refresh_label(paper)
            rows.append(widgets.VBox([widgets.HBox([swatch, master, expand, solo]), body]))

        btn_all = widgets.Button(description="Show all", layout=widgets.Layout(width="90px"))
        btn_none = widgets.Button(description="Hide all", layout=widgets.Layout(width="90px"))
        btn_all.on_click(lambda _: self.set_all(True))
        btn_none.on_click(lambda _: self.set_all(False))

        self.widget = widgets.VBox(
            [widgets.HTML("<b>Papers &amp; legend</b>"), widgets.HBox([btn_all, btn_none])] + rows,
            layout=widgets.Layout(width="400px", max_height="460px", overflow="auto",
                                  border="1px solid #666", padding="6px"),
        )

    # ---------- wiring ----------
    def _wire(self, paper, master, expand, solo, body):
        def on_master(change):
            if self._busy:
                return
            self._busy = True
            try:
                for cb in self.curve_cbs[paper].values():
                    cb.value = change["new"]
            finally:
                self._busy = False
            self._refresh_label(paper)
            self._fire()

        def on_child(change):
            if self._busy:
                return
            self._busy = True
            try:
                master.value = all(cb.value for cb in self.curve_cbs[paper].values())
            finally:
                self._busy = False
            self._refresh_label(paper)
            self._fire()

        def on_expand(change):
            body.layout.display = "flex" if change["new"] else "none"
            expand.description = "▾" if change["new"] else "▸"

        master.observe(on_master, names="value")
        for cb in self.curve_cbs[paper].values():
            cb.observe(on_child, names="value")
        expand.observe(on_expand, names="value")
        solo.on_click(lambda _: self.solo(paper))

    def _refresh_label(self, paper):
        on = sum(cb.value for cb in self.curve_cbs[paper].values())
        self.master[paper].description = f"{paper}  ({on}/{self._counts[paper]})"

    def _fire(self):
        if self._busy:
            return
        for fn in self._callbacks:
            fn(None)

    # ---------- bulk actions ----------
    def set_all(self, value):
        self._busy = True
        try:
            for paper in self.papers:
                for cb in self.curve_cbs[paper].values():
                    cb.value = value
                self.master[paper].value = value
                self._refresh_label(paper)
        finally:
            self._busy = False
        self._fire()

    def solo(self, only_paper):
        self._busy = True
        try:
            for paper in self.papers:
                flag = paper == only_paper
                for cb in self.curve_cbs[paper].values():
                    cb.value = flag
                self.master[paper].value = flag
                self._refresh_label(paper)
        finally:
            self._busy = False
        self._fire()

    # ---------- public API ----------
    def on_change(self, fn):
        self._callbacks.append(fn)

    def active_labels(self):
        return {lab for p in self.papers for lab, cb in self.curve_cbs[p].items() if cb.value}

    def active_papers(self):
        return [p for p in self.papers if any(cb.value for cb in self.curve_cbs[p].values())]

    def filter(self, d):
        return d[d["label"].isin(self.active_labels())]


def build_legend_selector(frames, palette_dict, marker_dict):
    return LegendSelector(frames, palette_dict, marker_dict)


In [19]:
#@title Block 3: Import CSVs and Rebuild Plot Data {display-mode: "form"}
import os, re, json, hashlib, shutil, tempfile, uuid
from pathlib import Path
import numpy as np
import pandas as pd

if 'historic_raw_list' not in globals() or 'historic_df_list' not in globals():
    raise RuntimeError('Run Block 1 first, then Block 2, then this block.')

MASTER_DIR = globals().get('MASTER_DIR', '/content/drive/MyDrive/TurbineResearch')
root = Path(MASTER_DIR)
uploads = root / 'undergrad_uploads'
archive = uploads / 'archive'
archive.mkdir(parents=True, exist_ok=True)
raw_file = root / 'undergrad_raw_master.parquet'
peak_file = root / 'undergrad_peak_master.parquet'

# Optional numeric fields remain unknown if not supplied. U is NOT mandatory.
NUMERIC = ['TSR','Cp','N','cToR','Re_D','Re_rot','alpha_p','U_m_s','Pressure_bar']
CONDITIONS = ['N','cToR','Re_D','Re_rot','alpha_p','U_m_s','Pressure_bar',
              'Pitch_direction','Surface_Finish','Foil_Profile','Struts','Strut_loss',
              'Medium','Curve_ID']

def read_curve(path):
    d = pd.read_csv(path, encoding='utf-8-sig')
    d.columns = [str(c).strip() for c in d.columns]
    if not {'TSR','Cp'}.issubset(d.columns):
        bare = pd.read_csv(path, header=None, encoding='utf-8-sig')
        # Accept a genuinely numeric two-column WPD export, retaining point 1.
        if bare.shape[1] != 2:
            raise ValueError('Missing TSR/Cp headers. Add exact headers TSR,Cp,N,cToR and conditions.')
        bare = bare.apply(pd.to_numeric, errors='raise')
        bare.columns = ['TSR','Cp']; d = bare
    if d.empty:
        raise ValueError('CSV has no data rows.')
    for col in d.columns:
        if d[col].dtype == object:
            d[col] = d[col].map(lambda v: v.strip() if isinstance(v,str) else v)
            d[col] = d[col].replace('', np.nan)
    for col, pattern in {'N':r'\bN\s*=\s*(\d+)',
                         'cToR':r'\bcToR\s*=\s*([\d.]+)'}.items():
        match = re.search(pattern, path.stem)
        if col not in d: d[col] = np.nan
        if match: d[col] = d[col].fillna(float(match.group(1)))
    for col in ['Re_D','alpha_p']:
        if col not in d: d[col] = np.nan
    for col in NUMERIC:
        if col in d:
            # Reject text units instead of silently turning a known value into NaN.
            d[col] = pd.to_numeric(d[col], errors='raise')
            if np.isinf(d[col]).any(): raise ValueError(f'{col} contains infinity.')
    if d[['TSR','Cp','N','cToR']].isna().any().any():
        raise ValueError('TSR,Cp,N,cToR must be present on every row; fill constants down.')
    if ((d.N <= 0) | (d.N % 1 != 0) | (d.cToR <= 0)).any():
        raise ValueError('N must be a positive integer and cToR must be positive.')
    if (d.TSR < 0).any():
        raise ValueError('Negative TSR found. Review the point before importing.')
    paper = path.stem.split('(')[0].split('[')[0].strip()
    if not paper: raise ValueError('Filename must start with the source name.')
    d['dataset_source'] = paper
    d['source_file'] = path.name
    d['ingest_file_key'] = path.name  # Reimporting this filename replaces its own rows.
    cols = [c for c in CONDITIONS if c in d]
    # Also split by additional supplied condition columns, excluding documentation.
    ignore = set(NUMERIC + CONDITIONS + ['solidity','Dynamic_Flow_Opacity','Dynamic_Blockage',
        'dataset_source','source_file','ingest_file_key','label','config_id','filename','Action',
        'Notes','Re_basis','Pitch_basis','Pitch_definition','original_dataset_source'])
    cols += sorted(c for c in d if c not in ignore and not c.startswith('Unnamed:')
                   and not c.startswith('assumed_'))
    def identify(row):
        values = {c: None if pd.isna(row[c]) else str(row[c]) for c in cols}
        token = hashlib.sha256(json.dumps(values, sort_keys=True).encode()).hexdigest()[:16]
        return path.stem + ' | ' + token
    d['config_id'] = d.apply(identify, axis=1)
    d['label'] = d['config_id']
    # This project's explicit circumference-based solidity convention.
    d['solidity'] = d.N * d.cToR / (2*np.pi)
    d['Dynamic_Flow_Opacity'] = d.solidity * d.TSR
    d['Dynamic_Blockage'] = d.Dynamic_Flow_Opacity
    if d.Re_D.isna().any(): print('    Re_D unknown for some rows; use Unmasked All Data to include them.', flush=True)
    if (d.Cp > .593).any(): print('    NOTE: Cp > 0.593; verify normalization/calibration.', flush=True)
    return d

def normalize_types(d):
    d = d.copy()
    for c in NUMERIC:
        if c in d: d[c] = pd.to_numeric(d[c], errors='raise')
    for c in d.select_dtypes(include=['object','string']).columns:
        d[c] = d[c].map(lambda v: None if pd.isna(v) else str(v))
    return d

def stage_parquet(d, target):
    fd, temporary = tempfile.mkstemp(prefix=target.stem+'_', suffix='.parquet', dir=root)
    os.close(fd)
    try: normalize_types(d).to_parquet(temporary, index=False)
    except Exception:
        os.unlink(temporary); raise
    return temporary

def combine(frames):
    frames = [d for d in frames if not d.empty]
    return pd.concat(frames, ignore_index=True, sort=False) if frames else pd.DataFrame()

print('Loading saved undergraduate data...', flush=True)
df_raw_cum = pd.read_parquet(raw_file) if raw_file.exists() else pd.DataFrame()
files = sorted(uploads.glob('*.csv'), key=lambda p:p.name.casefold())
print(f'Found {len(files)} CSV file(s) in {uploads}', flush=True)
accepted = []; failures = []
for number, path in enumerate(files,1):
    print(f'[{number}/{len(files)}] Reading {path.name}', flush=True)
    try:
        d = read_curve(path)
        if 'ingest_file_key' in df_raw_cum:
            df_raw_cum = df_raw_cum[df_raw_cum.ingest_file_key != path.name]
        df_raw_cum = combine([df_raw_cum,d])
        accepted.append(path)
        print(f'    Validated {len(d)} points / {d.config_id.nunique()} curve(s).', flush=True)
    except Exception as exc:
        failures.append(path.name)
        print(f'    ERROR: {exc}\n    Left in uploads for correction.', flush=True)

if not df_raw_cum.empty:
    if 'config_id' not in df_raw_cum or df_raw_cum.config_id.isna().any():
        raise ValueError('Saved undergraduate master has missing config_id values. No files archived; inspect the old master.')
    df_raw_cum = df_raw_cum.reset_index(drop=True)
    df_peak_cum = df_raw_cum.loc[df_raw_cum.groupby('config_id',sort=False).Cp.idxmax()].copy()
else:
    df_peak_cum = pd.DataFrame()

df_raw = combine(historic_raw_list + [df_raw_cum])
df_peaks = combine(historic_df_list + [df_peak_cum])
if df_raw.empty:
    print('No valid data available. Nothing saved or archived.', flush=True)
else:
    print('Saving undergraduate data and rebuilding BOTH plot master files...', flush=True)
    targets = [(df_raw_cum,raw_file),(df_peak_cum,peak_file),
               (df_raw,root/'master_raw_matrix.parquet'),
               (df_peaks,root/'master_peak_matrix.parquet')]
    staged = []
    try:
        # Finish all serialization before replacing any master. Upload files remain
        # available if saving fails; a rerun replaces each accepted file idempotently.
        for data,target in targets: staged.append((stage_parquet(data,target),target))
        for temporary,target in staged: os.replace(temporary,target)
    finally:
        for temporary,_ in staged:
            if os.path.exists(temporary): os.unlink(temporary)
    for path in accepted:
        try:
            destination = archive / path.name
            if destination.exists():
                backup_dir = archive / 'previous' / uuid.uuid4().hex
                backup_dir.mkdir(parents=True)
                shutil.move(str(destination),str(backup_dir/destination.name))
            shutil.move(str(path),str(destination))
            print(f'    Archived: {path.name}', flush=True)
        except Exception as exc:
            print(f'    Archive warning for {path.name}: {exc}. Data saved; rerunning is safe.',flush=True)
    print(f'DONE: {len(accepted)} accepted; {len(failures)} failed. '
          f'Master: {len(df_raw)} points / {len(df_peaks)} peaks.',flush=True)
    print('Now rerun Action 1 and Action 2 to refresh their widgets and plots.',flush=True)


Loading saved undergraduate data...
Found 2 CSV file(s) in /content/drive/MyDrive/TurbineResearch/undergrad_uploads
[1/2] Reading Migliore1980 (Fig 11) (N = 2) (cToR = 0.114) (C001).csv
    Validated 32 points / 1 curve(s).
[2/2] Reading Migliore1980 (Fig 11) (N = 2) (cToR = 0.260) (C002).csv
    Validated 12 points / 1 curve(s).
Saving undergraduate data and rebuilding BOTH plot master files...
    Archived: Migliore1980 (Fig 11) (N = 2) (cToR = 0.114) (C001).csv
    Archived: Migliore1980 (Fig 11) (N = 2) (cToR = 0.260) (C002).csv
DONE: 2 accepted; 0 failed. Master: 5160 points / 289 peaks.
Now rerun Action 1 and Action 2 to refresh their widgets and plots.


In [26]:
#@title Action 1: Interactive Domain Tessellation (TSR vs. Opacity) {display-mode: "form"}

import os
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output

MASTER_DIR = "/content/drive/MyDrive/TurbineResearch"
raw_path = os.path.join(MASTER_DIR, "master_raw_matrix.parquet")

if not os.path.exists(raw_path):
    print("Missing master_raw_matrix.parquet. Run Block 3 first.")

else:
    # ----------------------------------------------------------
    # 1. LOAD + SHARED CLEANUP (same prep as the storyboard cell)
    # ----------------------------------------------------------
    df = prepare_frames([pd.read_parquet(raw_path)])[0]

    if "Dynamic_Flow_Opacity" not in df.columns:
        raise ValueError("Missing Dynamic_Flow_Opacity. Run the ingestion and compilation cells first.")

    for column in ["TSR", "Dynamic_Flow_Opacity", "Re_D"]:
        df[column] = pd.to_numeric(df[column], errors="coerce")

    valid_coordinates = np.isfinite(
        df[["TSR", "Dynamic_Flow_Opacity"]].to_numpy(dtype=float)
    ).all(axis=1)

    excluded_count = int((~valid_coordinates).sum())
    df = df.loc[valid_coordinates].copy()

    if excluded_count:
        print(
            f"Excluded {excluded_count} rows with missing or invalid "
            "TSR/opacity coordinates from this view."
        )

    if df.empty:
        print("No valid TSR/opacity points are available.")

    else:
        # ------------------------------------------------------
        # 2. FIXED AXIS LIMITS
        # ------------------------------------------------------
        def padded_limits(series):
            low = float(series.min())
            high = float(series.max())
            span = high - low
            padding = span * 0.05 if span > 0 else max(abs(low) * 0.05, 0.1)
            return low - padding, high + padding

        global_xlim = padded_limits(df["TSR"])
        global_ylim = padded_limits(df["Dynamic_Flow_Opacity"])
        global_extent = [global_xlim[0], global_xlim[1], global_ylim[0], global_ylim[1]]

        valid_reds = df.loc[np.isfinite(df["Re_D"]) & (df["Re_D"] > 0), "Re_D"]

        if valid_reds.empty:
            min_red_scaled, max_red_scaled = 0.0, 10.0
        else:
            min_red_scaled = float(np.floor(valid_reds.min() / 1e4) / 10.0)
            max_red_scaled = float(np.ceil(valid_reds.max() / 1e4) / 10.0)
            if max_red_scaled <= min_red_scaled:
                max_red_scaled = min_red_scaled + 0.1

        # ------------------------------------------------------
        # 3. CONTROLS (shared legend panel: 1 checkbox per paper, expand for curves)
        # ------------------------------------------------------
        palette_dict, marker_dict = make_style_dicts(sorted(df["dataset_source"].unique()))
        legend = build_legend_selector([df], palette_dict, marker_dict)

        view_mode = widgets.RadioButtons(
            options=[
                "Combined Tessellation (Re_D Masked)",
                "Combined Tessellation (Unmasked All Data)",
                "Dataset Breakdown View (Re_D Masked)",
                "Dataset Breakdown View (Unmasked All Data)",
            ],
            value="Combined Tessellation (Unmasked All Data)",
            description="Layout:",
            layout=widgets.Layout(width="max-content"),
        )

        red_slider = widgets.FloatRangeSlider(
            value=[min_red_scaled, max_red_scaled],
            min=min_red_scaled,
            max=max_red_scaled,
            step=0.1,
            description="Re_D (×10⁵):",
            readout_format=".1f",
            continuous_update=False,
            style={"description_width": "initial"},
            layout=widgets.Layout(width="420px", display="none"),
        )

        out_plot = widgets.Output()

        # ------------------------------------------------------
        # 4. PLOTTING
        # ------------------------------------------------------
        def update_tessellation(*args):
            is_unmasked = "Unmasked" in view_mode.value
            is_combined = view_mode.value.startswith("Combined")
            red_slider.layout.display = "none" if is_unmasked else "flex"

            with out_plot:
                clear_output(wait=True)

                selected = legend.filter(df)

                if selected.empty:
                    print("Nothing selected. Tick a paper (or expand it and pick curves) in the legend panel.")
                    return

                if is_unmasked:
                    df_filtered = selected
                else:
                    red_low, red_high = red_slider.value
                    df_filtered = selected[
                        np.isfinite(selected["Re_D"])
                        & (selected["Re_D"] > 0)
                        & selected["Re_D"].between(red_low * 1e5, red_high * 1e5)
                    ].copy()

                    removed_count = len(selected) - len(df_filtered)
                    if removed_count:
                        print(
                            f"Re_D filter excludes {removed_count} points "
                            "with missing, invalid, or out-of-range Re_D."
                        )

                if df_filtered.empty:
                    print(
                        "No points match this selection. "
                        "Choose Unmasked All Data or adjust the Re_D range."
                    )
                    return

                fig_dummy, ax_dummy = plt.subplots()
                try:
                    dummy_hex = ax_dummy.hexbin(
                        df_filtered["TSR"], df_filtered["Dynamic_Flow_Opacity"],
                        gridsize=30, extent=global_extent, mincnt=1,
                    )
                    counts = dummy_hex.get_array()
                    global_vmax = float(counts.max()) if len(counts) else 1.0
                finally:
                    plt.close(fig_dummy)

                norm = mcolors.Normalize(vmin=0, vmax=max(1.0, global_vmax))

                if is_combined:
                    title = "All selected papers"
                    if not is_unmasked:
                        low, high = red_slider.value
                        title += f"\nRe_D: {low:g}–{high:g} × 10⁵"
                    plot_data = [(title, df_filtered)]
                    cols, rows = 1, 1
                    figsize = (7, 6.5)
                else:
                    shown_papers = [p for p in legend.papers if p in set(df_filtered["dataset_source"])]
                    plot_data = [
                        (source, df_filtered[df_filtered["dataset_source"] == source])
                        for source in shown_papers
                    ]
                    cols = min(3, len(plot_data))
                    rows = int(np.ceil(len(plot_data) / cols))
                    figsize = (5.5 * cols, 5.2 * rows)

                fig, axes = plt.subplots(
                    rows, cols, figsize=figsize, sharex=True, sharey=True,
                    squeeze=False, constrained_layout=True,
                )

                axes_list = axes.ravel()
                last_hex = None

                for i, (source, subset) in enumerate(plot_data):
                    ax = axes_list[i]

                    if subset.empty:
                        ax.text(0.5, 0.5, "No points within the Re_D filter",
                                ha="center", va="center", transform=ax.transAxes)
                    else:
                        last_hex = ax.hexbin(
                            subset["TSR"], subset["Dynamic_Flow_Opacity"],
                            gridsize=30, extent=global_extent, mincnt=1,
                            norm=norm, cmap="plasma", edgecolors="white", linewidths=0.5,
                        )

                    ax.set_title(source, fontsize=12, fontweight="bold")
                    ax.set_xlim(global_xlim)
                    ax.set_ylim(global_ylim)

                    ax.axhline(0, color="gray", linewidth=0.7, linestyle="--")
                    ax.axvline(0, color="gray", linewidth=0.7, linestyle="--")
                    ax.set_xlabel(r"Tip Speed Ratio ($\lambda$)")
                    ax.set_ylabel(r"Dynamic Flow Opacity ($\phi = \sigma\lambda$)")
                    ax.grid(True, linestyle=":", alpha=0.4)
                    ax.set_axisbelow(True)

                for ax in axes_list[len(plot_data):]:
                    ax.set_visible(False)

                if last_hex is not None:
                    colorbar = fig.colorbar(
                        last_hex, ax=list(axes_list[:len(plot_data)]),
                        fraction=0.035, pad=0.03,
                    )
                    colorbar.set_label("Digitized points per hexagon")

                fig.suptitle("Domain Tessellation — TSR vs. Flow Opacity", fontsize=15, fontweight="bold")
                plt.show()
                plt.close(fig)

                print(
                    f"Displaying {len(df_filtered):,} points from "
                    f"{df_filtered['dataset_source'].nunique()} papers "
                    f"({df_filtered['label'].nunique()} curves)."
                )

        # ------------------------------------------------------
        # 5. DISPLAY
        # ------------------------------------------------------
        legend.on_change(update_tessellation)
        view_mode.observe(update_tessellation, names="value")
        red_slider.observe(update_tessellation, names="value")

        controls = widgets.HBox([
            widgets.VBox([view_mode, red_slider]),
            legend.widget,
        ])

        display(controls, out_plot)
        update_tessellation()


📚 7 papers detected (from 7 raw dataset names).


Output()

In [27]:
#@title Action 2: 10-Phase Experimental Data LOESS Storyboard {display-mode: "form"}
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import os
import traceback
import re
import textwrap
from matplotlib.lines import Line2D
import warnings
warnings.filterwarnings('ignore')

# ------------------------------------------------------------------------------
# 1. DUAL-MATRIX LOADING
# ------------------------------------------------------------------------------
MASTER_DIR = '/content/drive/MyDrive/TurbineResearch'
raw_path = os.path.join(MASTER_DIR, 'master_raw_matrix.parquet')
peak_path = os.path.join(MASTER_DIR, 'master_peak_matrix.parquet')

if not os.path.exists(raw_path) or not os.path.exists(peak_path):
    print(f"❌ ERROR: Missing master matrices. Ensure Block 3 exported both files.")
else:
    df_raw = pd.read_parquet(raw_path)
    df_peaks = pd.read_parquet(peak_path)

    # Shared cleanup (same function the tessellation cell uses): merges messy dataset
    # names into one name per paper and gives every curve a unique, readable label.
    df_raw, df_peaks = prepare_frames([df_raw, df_peaks])

    # Colors / markers are per PAPER (all curves of a paper share them)
    available_datasets = sorted(set(df_raw['dataset_source']) | set(df_peaks['dataset_source']))
    palette_dict, marker_dict = make_style_dicts(available_datasets)

    # ------------------------------------------------------------------------------
    # 2. UI WIDGETS (shared paper selector)
    # ------------------------------------------------------------------------------
    out_plot = widgets.Output()
    phase_selector = widgets.Dropdown(
        options=[
            ('Sample Configs (Raw Sweeps)', 1),
            ('Sample Configs (Peak Isolated)', 2),
            ('Full Dataset (All Scatter Points)', 3),
            ('Full Dataset (All Sweeps)', 4),
            ('Full Dataset (Global LOESS Trends)', 5),
            ('Full Dataset (LOESS Breakdown by Blade Count)', 6),
            ('Peak Performance (Global Swarm & Count)', 7),
            ('Peak Performance (Global Polynomial Trends)', 8),
            ('Peak Performance (Architectural LOESS Breakdown)', 9),
            ('Peak Performance (Common Axis Overlay)', 10)
        ],
        value=1, description='Phase:', layout={'width': '450px'}
    )

    outlier_toggle = widgets.Checkbox(value=False, description='Remove Outliers (Z > 2.5)')
    negative_toggle = widgets.Checkbox(value=False, description='Remove Negative Cp')
    blind_toggle = widgets.Checkbox(value=False, description='Blind Datasets (Phase 5)')

    legend = build_legend_selector([df_raw, df_peaks], palette_dict, marker_dict)

    def get_filtered_data(z_filter, neg_filter):
        d_full = legend.filter(df_raw).copy()
        d_pks = legend.filter(df_peaks).copy()

        if neg_filter:
            d_full = d_full[d_full['Cp'] >= 0]
            d_pks = d_pks[d_pks['Cp'] >= 0]

        if z_filter and not d_pks.empty:
            z_scores = np.abs((d_pks['Cp'] - d_pks['Cp'].mean()) / d_pks['Cp'].std())
            valid_labels = d_pks[z_scores < 2.5]['label']
            d_full = d_full[d_full['label'].isin(valid_labels)]
            d_pks = d_pks[d_pks['label'].isin(valid_labels)]

        return d_full, d_pks

    def calc_poly_rmse(x, y, deg=2):
        if len(x) <= deg or len(np.unique(x)) <= deg: return np.nan
        poly = np.poly1d(np.polyfit(x, y, deg))
        return np.sqrt(np.mean((y - poly(x))**2))

    def update_storyboard(*args):
        step = phase_selector.value
        active_datasets = legend.active_papers()

        d_full, d_pks = get_filtered_data(outlier_toggle.value, negative_toggle.value)

        # Sample configs for phases 1-2: 5 random curves out of whatever is currently selected
        act_raw = legend.filter(df_raw)
        act_pks = legend.filter(df_peaks)
        avail_labels = sorted(act_raw['label'].unique())
        if avail_labels:
            rng = np.random.RandomState(42)
            sample_labels = rng.choice(avail_labels, min(5, len(avail_labels)), replace=False)
        else:
            sample_labels = []
        df_samples_raw = act_raw[act_raw['label'].isin(sample_labels)]
        df_samples_peaks = act_pks[act_pks['label'].isin(sample_labels)]

        with out_plot:
            clear_output(wait=True)
            title_dict = {v: k for k, v in phase_selector.options}
            display(HTML(f"<h3>Phase {step}: {title_dict[step]}</h3>"))

            if not active_datasets:
                print("⚠️ Nothing selected. Tick a paper (or expand it and pick curves) in the legend panel.")
                return

            y_ref = d_full['Cp'] if step <= 6 else d_pks['Cp']
            if len(y_ref) == 0:
                print("No data available with the current filters.")
                return

            pad_y = 0.05
            cp_range = max(0.1, y_ref.max() - y_ref.min())
            Y_LIM = (y_ref.min() - pad_y * cp_range, y_ref.max() + pad_y * cp_range)

            pad_x = 0.125
            tsr_range = df_raw['TSR'].max() - df_raw['TSR'].min()
            op_range = df_raw['Dynamic_Flow_Opacity'].max() - df_raw['Dynamic_Flow_Opacity'].min()
            X_LIMS_TSR = (df_raw['TSR'].min() - pad_x * tsr_range, df_raw['TSR'].max() + pad_x * tsr_range)
            X_LIMS_OP = (df_raw['Dynamic_Flow_Opacity'].min() - pad_x * op_range, df_raw['Dynamic_Flow_Opacity'].max() + pad_x * op_range)

            try:
                if step in [6, 9]:
                    fig, axes = plt.subplots(4, 2, figsize=(17, 18))
                else:
                    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(19, 6.5))

                label_tsr = r'Tip Speed Ratio ($\lambda$)'
                label_op = r'Dynamic Flow Opacity ($\phi = \sigma \cdot \lambda$)'
                colors = sns.color_palette("viridis", 4)

                # --- RAW SWEEP PHASES (1-6) ---
                if step == 1:
                    sns.lineplot(data=df_samples_raw[df_samples_raw['dataset_source'].isin(active_datasets)],
                                 x='TSR', y='Cp', hue='label', style='dataset_source', markers=marker_dict, dashes=False, alpha=0.6, ax=ax1)
                    sns.lineplot(data=df_samples_raw[df_samples_raw['dataset_source'].isin(active_datasets)],
                                 x='Dynamic_Flow_Opacity', y='Cp', hue='label', style='dataset_source', markers=marker_dict, dashes=False, alpha=0.6, ax=ax2)

                elif step == 2:
                    df_samp_raw_active = df_samples_raw[df_samples_raw['dataset_source'].isin(active_datasets)]
                    sns.lineplot(data=df_samp_raw_active, x='TSR', y='Cp', hue='label', legend=False, alpha=0.3, ax=ax1)
                    sns.lineplot(data=df_samp_raw_active, x='Dynamic_Flow_Opacity', y='Cp', hue='label', legend=False, alpha=0.3, ax=ax2)

                    sample_peaks_filtered = df_samples_peaks[df_samples_peaks['dataset_source'].isin(active_datasets)]
                    if negative_toggle.value:
                        sample_peaks_filtered = sample_peaks_filtered[sample_peaks_filtered['Cp'] >= 0]

                    sns.scatterplot(data=sample_peaks_filtered, x='TSR', y='Cp', style='dataset_source', hue='dataset_source',
                                    markers=marker_dict, palette=palette_dict, s=140, zorder=5, edgecolor='black', ax=ax1)
                    sns.scatterplot(data=sample_peaks_filtered, x='Dynamic_Flow_Opacity', y='Cp', style='dataset_source', hue='dataset_source',
                                    markers=marker_dict, palette=palette_dict, s=140, zorder=5, edgecolor='black', ax=ax2)

                elif step == 3:
                    sns.scatterplot(data=d_full, x='TSR', y='Cp', style='dataset_source', hue='dataset_source', markers=marker_dict, palette=palette_dict, alpha=0.4, ax=ax1)
                    sns.scatterplot(data=d_full, x='Dynamic_Flow_Opacity', y='Cp', style='dataset_source', hue='dataset_source', markers=marker_dict, palette=palette_dict, alpha=0.4, ax=ax2)

                elif step == 4:
                    d_lines = d_full.groupby('label').filter(lambda x: len(x) > 1)
                    if not d_lines.empty:
                        sns.lineplot(data=d_lines, x='TSR', y='Cp', units='label', estimator=None, alpha=0.1, ax=ax1, color='blue')
                        sns.lineplot(data=d_lines, x='Dynamic_Flow_Opacity', y='Cp', units='label', estimator=None, alpha=0.1, ax=ax2, color='blue')
                    else:
                        ax1.text(0.5, 0.5, "No multi-point sweeps available.", ha='center', va='center', transform=ax1.transAxes)
                        ax2.text(0.5, 0.5, "No multi-point sweeps available.", ha='center', va='center', transform=ax2.transAxes)

                elif step == 5:
                    if blind_toggle.value:
                        sns.scatterplot(data=d_full, x='TSR', y='Cp', color='gray', alpha=0.4, ax=ax1)
                        sns.scatterplot(data=d_full, x='Dynamic_Flow_Opacity', y='Cp', color='gray', alpha=0.4, ax=ax2)
                    else:
                        sns.scatterplot(data=d_full, x='TSR', y='Cp', style='dataset_source', hue='dataset_source', markers=marker_dict, palette=palette_dict, alpha=0.2, ax=ax1)
                        sns.scatterplot(data=d_full, x='Dynamic_Flow_Opacity', y='Cp', style='dataset_source', hue='dataset_source', markers=marker_dict, palette=palette_dict, alpha=0.2, ax=ax2)

                    sns.regplot(data=d_full, x='TSR', y='Cp', scatter=False, lowess=True, color='red', line_kws={'lw': 3}, ax=ax1)
                    sns.regplot(data=d_full, x='Dynamic_Flow_Opacity', y='Cp', scatter=False, lowess=True, color='red', line_kws={'lw': 3}, ax=ax2)

                elif step == 6:
                    for i, n in enumerate([1, 2, 3, 4]):
                        subset = d_full[d_full['N'] == n]
                        rmse_t = calc_poly_rmse(subset['TSR'], subset['Cp'])
                        rmse_b = calc_poly_rmse(subset['Dynamic_Flow_Opacity'], subset['Cp'])

                        sns.scatterplot(data=d_full, x='TSR', y='Cp', color='lightgray', alpha=0.1, ax=axes[i, 0], edgecolor=None)
                        sns.scatterplot(data=subset, x='TSR', y='Cp', style='dataset_source', hue='dataset_source', markers=marker_dict, palette=palette_dict, alpha=0.4, ax=axes[i, 0])
                        if len(subset) > 2: sns.regplot(data=subset, x='TSR', y='Cp', scatter=False, lowess=True, color='darkred', line_kws={'lw': 3}, ax=axes[i, 0])

                        sns.scatterplot(data=d_full, x='Dynamic_Flow_Opacity', y='Cp', color='lightgray', alpha=0.1, ax=axes[i, 1], edgecolor=None)
                        sns.scatterplot(data=subset, x='Dynamic_Flow_Opacity', y='Cp', style='dataset_source', hue='dataset_source', markers=marker_dict, palette=palette_dict, alpha=0.4, ax=axes[i, 1])
                        if len(subset) > 2: sns.regplot(data=subset, x='Dynamic_Flow_Opacity', y='Cp', scatter=False, lowess=True, color='darkred', line_kws={'lw': 3}, ax=axes[i, 1])

                        axes[i, 0].set_title(f'N={n}: Full Data (Poly RMSE: {rmse_t:.3f})', fontweight='bold')
                        axes[i, 1].set_title(f'N={n}: Full Data (Poly RMSE: {rmse_b:.3f})', fontweight='bold')
                        axes[i, 0].set_ylim(Y_LIM); axes[i, 1].set_ylim(Y_LIM)
                        axes[i, 0].set_xlim(X_LIMS_TSR); axes[i, 1].set_xlim(X_LIMS_OP)
                        axes[i, 0].grid(True, linestyle='--', alpha=0.5); axes[i, 1].grid(True, linestyle='--', alpha=0.5)
                        if i == 3:
                            axes[i, 0].set_xlabel(label_tsr)
                            axes[i, 1].set_xlabel(label_op)

                # --- PEAK DATA PHASES (7-10) ---
                elif step == 7:
                    sns.scatterplot(data=d_pks, x='TSR', y='Cp', hue='N', style='dataset_source', markers=marker_dict, palette='viridis', s=80, alpha=0.8, edgecolor='k', ax=ax1)
                    sns.scatterplot(data=d_pks, x='Dynamic_Flow_Opacity', y='Cp', hue='N', style='dataset_source', markers=marker_dict, palette='viridis', s=80, alpha=0.8, edgecolor='k', ax=ax2)
                    ax1.set_title(f'Peak Swarm (Configs mapped: {len(d_pks)})', fontweight='bold')
                    ax2.set_title(f'Peak Swarm (Configs mapped: {len(d_pks)})', fontweight='bold')

                elif step == 8:
                    rmse_t = calc_poly_rmse(d_pks['TSR'], d_pks['Cp'])
                    rmse_b = calc_poly_rmse(d_pks['Dynamic_Flow_Opacity'], d_pks['Cp'])

                    sns.scatterplot(data=d_pks, x='TSR', y='Cp', style='dataset_source', hue='dataset_source', markers=marker_dict, palette=palette_dict, alpha=0.6, ax=ax1)
                    sns.regplot(data=d_pks, x='TSR', y='Cp', scatter=False, order=2, color='black', line_kws={'lw': 2}, ax=ax1)
                    sns.scatterplot(data=d_pks, x='Dynamic_Flow_Opacity', y='Cp', style='dataset_source', hue='dataset_source', markers=marker_dict, palette=palette_dict, alpha=0.6, ax=ax2)
                    sns.regplot(data=d_pks, x='Dynamic_Flow_Opacity', y='Cp', scatter=False, order=2, color='black', line_kws={'lw': 2}, ax=ax2)

                    ax1.set_title(f'Peak Performance (Poly RMSE: {rmse_t:.3f})', fontweight='bold')
                    ax2.set_title(f'Peak Performance (Poly RMSE: {rmse_b:.3f})', fontweight='bold')

                elif step == 9:
                    for i, n in enumerate([1, 2, 3, 4]):
                        subset = d_pks[d_pks['N'] == n]
                        rmse_t = calc_poly_rmse(subset['TSR'], subset['Cp'])
                        rmse_b = calc_poly_rmse(subset['Dynamic_Flow_Opacity'], subset['Cp'])

                        sns.regplot(data=d_pks, x='TSR', y='Cp', scatter=False, color='gray', ax=axes[i, 0], line_kws={'linestyle': '--', 'alpha': 0.3}, order=2, ci=None)
                        sns.regplot(data=d_pks, x='Dynamic_Flow_Opacity', y='Cp', scatter=False, color='gray', ax=axes[i, 1], line_kws={'linestyle': '--', 'alpha': 0.3}, order=2, ci=None)

                        sns.scatterplot(data=d_pks, x='TSR', y='Cp', color='lightgray', alpha=0.3, ax=axes[i, 0], edgecolor=None)
                        sns.scatterplot(data=subset, x='TSR', y='Cp', style='dataset_source', hue='dataset_source', markers=marker_dict, palette=palette_dict, s=80, edgecolor='k', ax=axes[i, 0], legend=False)
                        if len(subset) > 2: sns.regplot(data=subset, x='TSR', y='Cp', scatter=False, lowess=True, color='darkred', line_kws={'lw': 2}, ax=axes[i, 0])

                        sns.scatterplot(data=d_pks, x='Dynamic_Flow_Opacity', y='Cp', color='lightgray', alpha=0.3, ax=axes[i, 1], edgecolor=None)
                        sns.scatterplot(data=subset, x='Dynamic_Flow_Opacity', y='Cp', style='dataset_source', hue='dataset_source', markers=marker_dict, palette=palette_dict, s=80, edgecolor='k', ax=axes[i, 1], legend=False)
                        if len(subset) > 2: sns.regplot(data=subset, x='Dynamic_Flow_Opacity', y='Cp', scatter=False, lowess=True, color='darkred', line_kws={'lw': 2}, ax=axes[i, 1])

                        axes[i, 0].set_title(f'N={n}: Peaks (Poly RMSE: {rmse_t:.3f})', fontweight='bold')
                        axes[i, 1].set_title(f'N={n}: Peaks (Poly RMSE: {rmse_b:.3f})', fontweight='bold')
                        axes[i, 0].set_ylim(Y_LIM); axes[i, 1].set_ylim(Y_LIM)
                        axes[i, 0].set_xlim(X_LIMS_TSR); axes[i, 1].set_xlim(X_LIMS_OP)
                        axes[i, 0].grid(True, linestyle='--', alpha=0.5); axes[i, 1].grid(True, linestyle='--', alpha=0.5)
                        if i == 3:
                            axes[i, 0].set_xlabel(label_tsr)
                            axes[i, 1].set_xlabel(label_op)

                elif step == 10:
                    for i, n in enumerate([1, 2, 3, 4]):
                        subset = d_pks[d_pks['N'] == n]
                        if subset.empty: continue

                        sns.scatterplot(data=subset, x='TSR', y='Cp', style='dataset_source', hue='dataset_source', markers=marker_dict, palette=palette_dict, alpha=0.6, s=70, edgecolor='k', ax=ax1)
                        if len(subset) > 2: sns.regplot(data=subset, x='TSR', y='Cp', scatter=False, lowess=True, color=colors[i], line_kws={'lw': 2, 'label': f'N={n}'}, ax=ax1)

                        sns.scatterplot(data=subset, x='Dynamic_Flow_Opacity', y='Cp', style='dataset_source', hue='dataset_source', markers=marker_dict, palette=palette_dict, alpha=0.6, s=70, edgecolor='k', ax=ax2)
                        if len(subset) > 2: sns.regplot(data=subset, x='Dynamic_Flow_Opacity', y='Cp', scatter=False, lowess=True, color=colors[i], line_kws={'lw': 2, 'label': f'N={n}'}, ax=ax2)

                    ax1.set_title('Common Axis: Peak Cp vs. TSR', fontweight='bold')
                    ax2.set_title('Common Axis: Peak Cp vs. Opacity', fontweight='bold')

                # --- GLOBAL FORMATTING & LEGEND RELOCATION ---
                if step not in [6, 9]:
                    ax1.set_xlabel(label_tsr)
                    ax1.set_ylabel(r'Power Coefficient ($C_P$)')
                    ax2.set_xlabel(label_op)
                    ax2.set_ylabel('')

                    ax1.set_ylim(Y_LIM); ax2.set_ylim(Y_LIM)
                    ax1.set_xlim(X_LIMS_TSR); ax2.set_xlim(X_LIMS_OP)
                    ax1.grid(True, linestyle=':', alpha=0.6); ax2.grid(True, linestyle=':', alpha=0.6)

                    if step < 7:
                        ax1.set_title(ax1.get_title() or "Full Dataset: TSR vs Efficiency", fontweight='bold')
                        ax2.set_title(ax2.get_title() or "Full Dataset: Flow Opacity vs Efficiency", fontweight='bold')

                # --- SIDE LEGEND: one tidy column to the right of the plots ---
                handles, labels = [], []
                for ax in fig.axes:
                    h, l = ax.get_legend_handles_labels()
                    handles.extend(h)
                    labels.extend(l)
                    if ax.get_legend() is not None:
                        ax.get_legend().remove()

                by_label = dict(zip(labels, handles))
                dict_N, dict_papers, dict_curves = {}, {}, {}
                for l, h in by_label.items():
                    if l in ['N', 'dataset_source', 'label']:
                        continue
                    if str(l) in ['1', '2', '3', '4', '1.0', '2.0', '3.0', '4.0']:
                        dict_N[f'N={int(float(l))}'] = h
                    elif re.fullmatch(r'N=\d+', str(l)):
                        dict_N[str(l)] = h
                    elif l in palette_dict:
                        dict_papers[l] = h
                    else:
                        dict_curves[l] = h

                entry_handles, entry_labels = [], []
                for title, dct in (('Blade count', dict_N), ('Papers', dict_papers), ('Curves', dict_curves)):
                    if dct:
                        entry_handles.append(Line2D([], [], linestyle='none', marker='None'))
                        entry_labels.append(f'— {title} —')
                        for l, h in dct.items():
                            entry_handles.append(h)
                            entry_labels.append(textwrap.fill(str(l), 36))

                n_lines = sum(l.count('\n') + 1 for l in entry_labels)
                w_in, h_in = fig.get_size_inches()
                fig.set_size_inches(w_in, max(h_in, 0.24 * n_lines + 1.2))

                plt.tight_layout(rect=[0, 0, 0.79, 1])
                if entry_handles:
                    fig.legend(entry_handles, entry_labels, loc='upper left',
                               bbox_to_anchor=(0.80, 0.985), ncol=1, frameon=True,
                               fontsize=9, borderaxespad=0.0, labelspacing=0.5)

                plt.show()

            except Exception as e:
                clear_output(wait=False)
                print(f"❌ Plotting Execution Error in Phase {step}: {e}")
                traceback.print_exc()

    phase_selector.observe(update_storyboard, names='value')
    outlier_toggle.observe(update_storyboard, names='value')
    negative_toggle.observe(update_storyboard, names='value')
    blind_toggle.observe(update_storyboard, names='value')
    legend.on_change(update_storyboard)

    controls = widgets.HBox([widgets.VBox([phase_selector, outlier_toggle, negative_toggle, blind_toggle]), legend.widget])
    display(controls, out_plot)
    update_storyboard(None)


📚 7 papers detected (from 7 raw dataset names).


Output()

In [25]:
#@title Action 3: Dataset Manager {display-mode: "form"}
import os
import shutil
import tempfile
from pathlib import Path
from datetime import datetime
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output

# Disable any old deletion widget when this cell is rerun.
for control in globals().get("_delete_controls", []):
    control.disabled = True


def make_delete_widget():
    root = Path("/content/drive/MyDrive/TurbineResearch")
    names = [
        "undergrad_raw_master.parquet",
        "undergrad_peak_master.parquet",
        "master_raw_matrix.parquet",
        "master_peak_matrix.parquet",
    ]

    paper = widgets.Dropdown(
        description="Paper:", options=[("Choose paper", None)],
        layout=widgets.Layout(width="95%")
    )
    curve = widgets.Dropdown(
        description="Curve:", options=[("Choose paper first", None)],
        layout=widgets.Layout(width="95%")
    )
    refresh = widgets.Button(description="Refresh list")
    preview = widgets.Button(description="Preview deletion")
    confirm = widgets.Checkbox(
        description="I checked the preview and want to delete this data.",
        value=False, disabled=True, indent=False,
        layout=widgets.Layout(width="95%")
    )
    delete = widgets.Button(
        description="Delete selected", button_style="danger", disabled=True
    )
    output = widgets.Output()
    state = {"raw": pd.DataFrame(), "pending": None}

    def reset(*_):
        state["pending"] = None
        confirm.value = False
        confirm.disabled = True
        delete.disabled = True

    def stamp(path):
        if not path.exists():
            return None
        stat = path.stat()
        return stat.st_size, stat.st_mtime_ns

    def choose_paper(*_):
        reset()
        raw = state["raw"]
        if paper.value is None:
            curve.options = [("Choose paper first", None)]
            return
        rows = raw[raw["dataset_source"].astype(str).eq(paper.value)]
        ids = sorted(rows["config_id"].dropna().astype(str).unique())
        curve.options = (
        [("All curves in this paper", "__ALL__")]
        + [(value, value) for value in ids]
        )

    def reload(*_):
        reset()
        raw_path = root / names[0]
        raw = pd.read_parquet(raw_path) if raw_path.exists() else pd.DataFrame()
        if not raw.empty:
            required = {"dataset_source", "config_id"}
            if not required.issubset(raw.columns):
                raise ValueError("Imported master is missing dataset_source or config_id.")
        state["raw"] = raw
        sources = (
            sorted(raw["dataset_source"].dropna().astype(str).unique())
            if not raw.empty else []
        )
        paper.options = [("Choose paper", None)] + [(s, s) for s in sources]
        paper.value = None
        choose_paper()

    def show_preview(_):
        reset()
        with output:
            clear_output(wait=True)
            if paper.value is None or curve.value is None:
                print("Choose a paper first.")
                return
            try:
                raw = pd.read_parquet(root / names[0])
                selected = raw["dataset_source"].astype(str).eq(paper.value)
                if curve.value != "__ALL__":
                    selected &= raw["config_id"].astype(str).eq(curve.value)
                rows = raw.loc[selected]
                ids = set(rows["config_id"].dropna().astype(str))
                if not ids:
                    print("No matching data. Click Refresh list.")
                    return

                tables, stamps = {}, {}
                for name in names:
                    path = root / name
                    stamps[name] = stamp(path)
                    if not path.exists():
                        continue
                    data = pd.read_parquet(path)
                    if not data.empty and "config_id" not in data:
                        raise ValueError(f"{name} has no config_id.")
                    mask = (
                        data["config_id"].astype(str).isin(ids)
                        if "config_id" in data
                        else pd.Series(False, index=data.index)
                    )
                    tables[name] = data.loc[~mask].copy()
                    print(f"{name}: {mask.sum()} rows to delete")

                columns = [
                    c for c in ["source_file", "config_id", "N", "cToR",
                                "Re_D", "alpha_p"] if c in rows
                ]
                display(rows[columns].drop_duplicates())
                state["pending"] = (tables, stamps)
                confirm.disabled = False
            except Exception as exc:
                print(f"Preview stopped: {exc}")

    def enable_delete(*_):
        delete.disabled = not (
            confirm.value and state["pending"] is not None
        )

    def delete_selected(_):
        if not confirm.value or state["pending"] is None:
            return

        tables, stamps = state["pending"]
        reset()  # Consume confirmation so another click cannot repeat deletion.
        staged, replaced = {}, []
        backup = None

        with output:
            clear_output(wait=True)
            try:
                if any(stamp(root / n) != s for n, s in stamps.items()):
                    raise RuntimeError("Data changed. Preview the selection again.")

                backup_root = root / "deletion_backups"
                backup_root.mkdir(parents=True, exist_ok=True)
                backup = Path(tempfile.mkdtemp(
                    prefix=datetime.now().strftime("%Y%m%d_%H%M%S_"),
                    dir=backup_root
                ))

                # Back up and prepare every table before replacing any.
                for name, data in tables.items():
                    shutil.copy2(root / name, backup / name)
                    fd, temporary = tempfile.mkstemp(suffix=".parquet", dir=root)
                    os.close(fd)
                    staged[name] = temporary
                    data.to_parquet(temporary, index=False)

                for name, temporary in staged.items():
                    os.replace(temporary, root / name)
                    replaced.append(name)

            except Exception as exc:
                for name in replaced:
                    shutil.copy2(backup / name, root / name)
                print(f"Deletion stopped: {exc}")
                if replaced:
                    print("Changed tables restored from backup.")
                return
            finally:
                for temporary in staged.values():
                    if os.path.exists(temporary):
                        os.unlink(temporary)

            reload()
            print("Selected data deleted. Archived CSVs are retained.")
            print(f"Backup saved in: {backup}")
            print("Rerun Action 1 and Action 2 to refresh your graphs.")

    paper.observe(choose_paper, names="value")
    curve.observe(reset, names="value")
    confirm.observe(enable_delete, names="value")
    refresh.on_click(reload)
    preview.on_click(show_preview)
    delete.on_click(delete_selected)

    reload()
    display(widgets.VBox([
        widgets.HTML("<b>Delete imported data</b>"),
        paper, curve,
        widgets.HBox([refresh, preview]),
        confirm, delete, output
    ]))
    return [paper, curve, refresh, preview, confirm, delete]


_delete_controls = make_delete_widget()